# 13d - CLIP-SENet CityFlowV2 Association

Runs CPU-only Stages 4-5 on CityFlowV2 using tracklet-level score fusion. The existing 10b checkpoint already contains TransReID 09v primary tracklet embeddings and DINOv2 tertiary tracklet embeddings from the current 10c v15 baseline. The 13c CLIP-SENet output is per detection, so this notebook validates tracklet keys against the Stage 4 embedding index, mean-pools detection embeddings to the Stage 4 tracklet order, applies PCA whitening plus L2 normalization, and injects them as the secondary score stream.

Fusion sweep: `sim = (1 - w_cs - w_dino) * sim_transreid + w_dino * sim_dinov2 + w_cs * sim_clipsenet`, where `w_dino = (1 - w_cs) * 0.60`. Therefore `w_cs=0.0` preserves the locked 10c v15 CLIP+DINOv2 baseline and should reproduce MTMC IDF1 about 0.7703.

## 1. Environment And Repository

In [ ]:
import os
import sys
import json
import time
import tarfile
import shutil
import subprocess
from pathlib import Path

if shutil.which("nvidia-smi"):
    raise RuntimeError("This notebook must run with enable_gpu=false. GPU is reserved for training kernels.")

WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "feature/pretrained-ensemble"


def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ({BRANCH})")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-cpu")
    except Exception:
        pip("faiss-gpu")

try:
    import trackeval
    print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip("motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click", "numpy", "scipy", "pandas", "scikit-learn")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], cwd=str(PROJECT))

print(f"Python: {sys.version.split()[0]}")
print(f"Project: {PROJECT}")
print("CPU-only environment ready")

## 2. Load 10b Checkpoint

In [ ]:
from src.core.io_utils import load_embeddings, load_tracklets_by_camera

PREV_SLUG = "mtmc-10b-stage-3-faiss-indexing"


def find_input_dir(slug: str, hints=()) -> Path:
    root = Path("/kaggle/input")
    direct = root / slug
    if direct.exists():
        return direct
    for path in root.iterdir():
        name = path.name.lower()
        if slug.lower() in name or any(hint.lower() in name for hint in hints):
            return path
    return direct


PREV_INPUT = find_input_dir(PREV_SLUG, hints=("10b", "stage-3", "faiss"))
checkpoint = PREV_INPUT / "checkpoint.tar.gz"

if not checkpoint.exists():
    print(f"checkpoint.tar.gz not found at {checkpoint}; trying Kaggle API fallback")
    dl_dir = Path("/tmp/kaggle_10b_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    candidates = [
        "yahiaakhalafallah/mtmc-10b-stage-3-faiss-indexing",
        "gumfreddy/mtmc-10b-stage-3-faiss-indexing",
    ]
    for candidate in candidates:
        result = subprocess.run(
            ["kaggle", "kernels", "output", candidate, "--file", "checkpoint.tar.gz", "-p", str(dl_dir)],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint = dl_dir / "checkpoint.tar.gz"
        if checkpoint.exists():
            break

assert checkpoint.exists(), f"checkpoint.tar.gz not found for {PREV_SLUG}"

EXTRACT_DIR = Path("/tmp/pipeline_run")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json", encoding="utf-8") as handle:
    meta = json.load(handle)

RUN_NAME = meta["run_name"]
DATA_OUT = EXTRACT_DIR
RUN_DIR = EXTRACT_DIR / RUN_NAME
STAGE1_DIR = RUN_DIR / "stage1"
STAGE2_DIR = RUN_DIR / "stage2"
STAGE3_DIR = RUN_DIR / "stage3"
PRIMARY_EMBEDDINGS_PATH = STAGE2_DIR / "embeddings.npy"
TERTIARY_DINOV2_PATH = STAGE2_DIR / "embeddings_tertiary.npy"

assert PRIMARY_EMBEDDINGS_PATH.exists(), PRIMARY_EMBEDDINGS_PATH
assert TERTIARY_DINOV2_PATH.exists(), (
    f"Missing 10c v15 DINOv2 tertiary embeddings at {TERTIARY_DINOV2_PATH}. "
    "Do not run this sweep without the 0.7703 baseline input."
)

primary_embeddings, embedding_index = load_embeddings(STAGE2_DIR)
tracklets_by_camera = load_tracklets_by_camera(STAGE1_DIR)
tertiary_embeddings = __import__("numpy").load(TERTIARY_DINOV2_PATH)

assert primary_embeddings.shape[0] == len(embedding_index)
assert tertiary_embeddings.shape[0] == len(embedding_index)

repo_gt = PROJECT / "data" / "raw" / "cityflowv2"
dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
if any((repo_gt / cam / "gt" / "gt.txt").exists() for cam in ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]):
    GT_DIR = str(repo_gt)
elif dataset_gt.exists():
    GT_DIR = str(dataset_gt)
else:
    GT_DIR = ""

print(f"Run name: {RUN_NAME}")
print(f"Stage 2 primary: {primary_embeddings.shape} -> {PRIMARY_EMBEDDINGS_PATH}")
print(f"Stage 2 DINOv2 tertiary: {tertiary_embeddings.shape} -> {TERTIARY_DINOV2_PATH}")
print(f"Embedding index rows: {len(embedding_index)}")
print(f"Stage 1 cameras: {sorted(tracklets_by_camera)}")
print(f"GT dir: {GT_DIR or 'not found'}")

## 3. Load And Verify 13c CLIP-SENet Features

In [ ]:
import numpy as np

CLIP_SOURCE_SLUGS = [
    "13c-clip-senet-cityflowv2-features",
]
CLIP_KERNEL_IDS = [
    "yahiaakhalafallah/13c-clip-senet-cityflowv2-features",
]


def find_clip_feature_dir() -> Path:
    for slug in CLIP_SOURCE_SLUGS:
        candidate = Path("/kaggle/input") / slug / "clip_senet_features"
        if candidate.exists():
            return candidate
    for path in Path("/kaggle/input").iterdir():
        name = path.name.lower()
        if "13c" in name and "clip" in name and "senet" in name:
            candidate = path / "clip_senet_features"
            if candidate.exists():
                return candidate
    dl_dir = Path("/tmp/kaggle_13c_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for kernel_id in CLIP_KERNEL_IDS:
        print(f"Trying Kaggle API fallback for {kernel_id}")
        result = subprocess.run(["kaggle", "kernels", "output", kernel_id, "-p", str(dl_dir)], capture_output=True, text=True)
        print(result.stdout)
        print(result.stderr)
        for candidate in [dl_dir / "clip_senet_features", *dl_dir.rglob("clip_senet_features")]:
            if candidate.exists():
                return candidate
    raise FileNotFoundError("Could not locate 13c clip_senet_features output")


CLIP_FEATURE_DIR = find_clip_feature_dir()
manifest_path = CLIP_FEATURE_DIR / "manifest.json"
assert manifest_path.exists(), manifest_path
with open(manifest_path, encoding="utf-8") as handle:
    clip_manifest = json.load(handle)

print(f"CLIP-SENet feature dir: {CLIP_FEATURE_DIR}")
print(json.dumps(clip_manifest.get("summary", {}), indent=2))

camera_npz = {}
for camera_id in sorted(tracklets_by_camera):
    npz_path = CLIP_FEATURE_DIR / f"{camera_id}.npz"
    assert npz_path.exists(), f"Missing CLIP-SENet NPZ for {camera_id}: {npz_path}"
    camera_npz[camera_id] = np.load(npz_path)


source_keys_by_camera = {}
for row in embedding_index:
    key = (row["camera_id"], int(row["track_id"]))
    source_keys_by_camera.setdefault(row["camera_id"], set()).add(key)


def clip_tracklet_keys(camera_id: str, data: np.lib.npyio.NpzFile) -> set[tuple[str, int]]:
    track_ids = data["track_ids"].astype(np.int64)
    return {(camera_id, int(track_id)) for track_id in np.unique(track_ids)}


alignment_report = {}
alignment_problems = {}
for camera_id, data in camera_npz.items():
    source_keys = source_keys_by_camera.get(camera_id, set())
    clip_keys = clip_tracklet_keys(camera_id, data)
    missing_from_clip = sorted(source_keys - clip_keys)
    extra_in_clip = sorted(clip_keys - source_keys)

    alignment_report[camera_id] = {
        "detections": int(data["frame_ids"].shape[0]),
        "source_tracklets": int(len(source_keys)),
        "clip_tracklets": int(len(clip_keys)),
        "feature_dim": int(data["embeddings"].shape[1]),
        "missing_from_clip": missing_from_clip[:10],
        "extra_in_clip": extra_in_clip[:10],
    }
    if missing_from_clip or extra_in_clip:
        alignment_problems[camera_id] = alignment_report[camera_id]

if alignment_problems:
    print(json.dumps(alignment_problems, indent=2))
    raise RuntimeError("13c tracklet alignment failed; CLIP-SENet and Stage 4 tracklet key sets differ")

print("13c tracklet key alignment verified against Stage 4 embedding index")
print(json.dumps(alignment_report, indent=2))

## 4. Aggregate CLIP-SENet To Tracklet-Level Embeddings

In [ ]:
from src.stage2_features.embeddings import l2_normalize
from src.stage2_features.pca_whitening import PCAWhitener

clip_by_camera_track = {}
for camera_id, data in camera_npz.items():
    embeddings = l2_normalize(data["embeddings"].astype(np.float32))
    track_ids = data["track_ids"].astype(np.int64)
    for track_id in np.unique(track_ids):
        mask = track_ids == track_id
        pooled = embeddings[mask].mean(axis=0)
        norm = np.linalg.norm(pooled)
        if norm <= 1e-8:
            raise RuntimeError(f"Zero pooled CLIP-SENet embedding for {camera_id} track {int(track_id)}")
        clip_by_camera_track[(camera_id, int(track_id))] = (pooled / norm).astype(np.float32)

source_key_set = {(row["camera_id"], int(row["track_id"])) for row in embedding_index}
extra_clip_keys = sorted(set(clip_by_camera_track) - source_key_set)
if extra_clip_keys:
    print(f"Skipping {len(extra_clip_keys)} CLIP-SENet tracklets not present in Stage 4 source rows: {extra_clip_keys[:10]}")

clip_rows = []
missing = []
for row in embedding_index:
    key = (row["camera_id"], int(row["track_id"]))
    value = clip_by_camera_track.get(key)
    if value is None:
        missing.append(key)
    else:
        clip_rows.append(value)

if missing:
    print(f"Missing {len(missing)} CLIP-SENet tracklets required by Stage 4 source rows: {missing[:10]}")
    raise RuntimeError(f"Missing CLIP-SENet tracklet embeddings for {len(missing)} Stage 2 rows")

clip_raw = np.stack(clip_rows, axis=0).astype(np.float32)
clip_raw = l2_normalize(clip_raw)
raw_path = STAGE2_DIR / "embeddings_clip_senet_raw_tracklet.npy"
np.save(raw_path, clip_raw)

pca_components = min(384, clip_raw.shape[0], clip_raw.shape[1])
if pca_components < 50:
    raise RuntimeError(f"Too few CLIP-SENet tracklets for stable PCA: {clip_raw.shape}")

whitener = PCAWhitener(n_components=pca_components)
clip_pca = whitener.fit_transform(clip_raw)
clip_pca = l2_normalize(clip_pca)

CLIP_TRACKLET_EMBEDDINGS_PATH = STAGE2_DIR / "embeddings_clip_senet.npy"
CLIP_PCA_PATH = STAGE2_DIR / "pca_clip_senet_tracklet.pkl"
np.save(CLIP_TRACKLET_EMBEDDINGS_PATH, clip_pca.astype(np.float32))
whitener.save(CLIP_PCA_PATH)

assert clip_pca.shape[0] == primary_embeddings.shape[0] == tertiary_embeddings.shape[0]
print(f"CLIP-SENet raw tracklet embeddings: {clip_raw.shape} -> {raw_path}")
print(f"CLIP-SENet PCA/L2 tracklet embeddings: {clip_pca.shape} -> {CLIP_TRACKLET_EMBEDDINGS_PATH}")
print(f"PCA model: {CLIP_PCA_PATH}")
print("Fusion granularity decision: tracklet-level score fusion, matching Stage 4 primary and DINOv2 streams")

## 5. Locked 10c v15 Config

In [ ]:
AQE_K = 3
SIM_THRESH = 0.50
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70

APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
FIC_REG = 0.50
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38

INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30

BASELINE_DINOV2_WEIGHT = 0.60
WEIGHT_SWEEP = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]

MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False

print("Locked config:")
print(f"  AQE_K={AQE_K}, sim_thresh={SIM_THRESH}, algorithm={ALGORITHM}")
print(f"  appearance={APPEARANCE_WEIGHT}, hsv={HSV_WEIGHT}, spatiotemporal={ST_WEIGHT}")
print(f"  fic_reg={FIC_REG}, gallery={GALLERY_THRESH}, orphan={ORPHAN_MATCH_THRESH}")
print(f"  baseline DINOv2 weight at w_cs=0.0: {BASELINE_DINOV2_WEIGHT}")
print(f"  sweep: {WEIGHT_SWEEP}")

## 6. Run Fusion Sweep

In [ ]:
def secondary_embedding_overrides(weight: float) -> list[str]:
    if weight <= 0.0:
        return [
            "--override", "stage4.association.secondary_embeddings.path=",
            "--override", "stage4.association.secondary_embeddings.weight=0.0",
        ]
    return [
        "--override", f"stage4.association.secondary_embeddings.path={CLIP_TRACKLET_EMBEDDINGS_PATH}",
        "--override", f"stage4.association.secondary_embeddings.weight={weight}",
    ]


def tertiary_embedding_overrides(weight: float) -> list[str]:
    if weight <= 0.0:
        return [
            "--override", "stage4.association.tertiary_embeddings.path=",
            "--override", "stage4.association.tertiary_embeddings.weight=0.0",
        ]
    return [
        "--override", f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
        "--override", f"stage4.association.tertiary_embeddings.weight={weight}",
    ]


def ensure_upstream_links(run_name: str) -> Path:
    run_dir = DATA_OUT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    for stage_name in ("stage1", "stage2", "stage3"):
        src = RUN_DIR / stage_name
        dst = run_dir / stage_name
        if dst.exists():
            continue
        try:
            dst.symlink_to(src, target_is_directory=True)
        except OSError:
            shutil.copytree(src, dst, dirs_exist_ok=True)
    return run_dir


def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }


def count_clusters(run_name: str) -> int | None:
    path = DATA_OUT / run_name / "stage4" / "global_trajectories.json"
    if not path.exists():
        return None
    return len(json.loads(path.read_text(encoding="utf-8")))


def build_cmd(run_name: str, w_cs: float) -> tuple[list[str], float, float, float]:
    w_dino = round((1.0 - w_cs) * BASELINE_DINOV2_WEIGHT, 6)
    w_primary = round(1.0 - w_cs - w_dino, 6)
    if w_primary < -1e-8:
        raise ValueError(f"Invalid weights: w_primary={w_primary}, w_dino={w_dino}, w_cs={w_cs}")
    cmd = [
        sys.executable, "scripts/run_pipeline.py",
        "--config", "configs/default.yaml",
        "--dataset-config", "configs/datasets/cityflowv2.yaml",
        "--stages", "4,5",
        "--override", f"project.run_name={run_name}",
        "--override", f"project.output_dir={DATA_OUT}",
        "--override", f"stage4.association.query_expansion.k={AQE_K}",
        "--override", "stage4.association.query_expansion.alpha=5.0",
        "--override", "stage4.association.query_expansion.dba=false",
        "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
        "--override", f"stage4.association.solver={SOLVER}",
        "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
        "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
        "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
        "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
        "--override", f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
        "--override", f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
        "--override", f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
        "--override", "stage4.association.mutual_nn.top_k_per_query=20",
        "--override", "stage4.association.fic.enabled=true",
        "--override", f"stage4.association.fic.regularisation={FIC_REG}",
        "--override", f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
        "--override", f"stage4.association.reranking.k1={RERANKING_K1}",
        "--override", f"stage4.association.reranking.k2={RERANKING_K2}",
        "--override", f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
        "--override", f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
        "--override", "stage4.association.fac.enabled=false",
        "--override", f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
        "--override", f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
        "--override", f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
        "--override", f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
        "--override", f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
        "--override", f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
        "--override", f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
        "--override", f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
        *secondary_embedding_overrides(w_cs),
        *tertiary_embedding_overrides(w_dino),
        "--override", f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
        "--override", f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
        "--override", f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
        "--override", "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
        "--override", f"stage4.association.zone_model.bonus={ZONE_BONUS}",
        "--override", f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
        "--override", f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
        "--override", f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
        "--override", f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
        "--override", f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
        "--override", "stage4.association.hierarchical.max_merge_size=12",
        "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
        "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
        "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
        "--override", "stage4.association.gallery_expansion.enabled=true",
        "--override", f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
        "--override", f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
        "--override", "stage4.association.weights.length_weight_power=0.3",
        "--override", "stage4.association.temporal_overlap.enabled=true",
        "--override", "stage4.association.temporal_overlap.bonus=0.05",
        "--override", "stage4.association.temporal_overlap.max_mean_time=5.0",
        "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
        "--override", "stage5.stationary_filter.enabled=true",
        "--override", "stage5.stationary_filter.min_displacement_px=150",
        "--override", "stage5.stationary_filter.max_mean_velocity_px=2.0",
        "--override", "stage5.min_submission_confidence=0.15",
        "--override", "stage5.cross_id_nms_iou=0.40",
        "--override", "stage5.min_trajectory_confidence=0.30",
        "--override", "stage5.min_trajectory_frames=40",
        "--override", "stage5.track_edge_trim.enabled=false",
        "--override", "stage5.track_smoothing.enabled=false",
        "--override", "stage5.gt_frame_clip=true",
        "--override", "stage5.gt_zone_filter=true",
    ]
    if GT_DIR:
        cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]
    return cmd, w_primary, w_dino, w_cs


fusion_results = []
for w_cs in WEIGHT_SWEEP:
    label = f"clipsenet_{int(round(w_cs * 100)):03d}"
    run_name = f"{RUN_NAME}-{label}"
    ensure_upstream_links(run_name)
    cmd, w_primary, w_dino, w_clip_senet = build_cmd(run_name, w_cs)
    print("=" * 100)
    print(f"[{label}] w_primary={w_primary:.3f} w_dino={w_dino:.3f} w_cs={w_clip_senet:.3f}")
    print("CMD:", " ".join(str(part) for part in cmd))
    start = time.time()
    result = subprocess.run(cmd, cwd=str(PROJECT), capture_output=True, text=True)
    elapsed_min = (time.time() - start) / 60.0
    report_path = DATA_OUT / run_name / "stage5" / "evaluation_report.json"
    metrics = load_metrics(report_path)
    row = {
        "label": label,
        "run_name": run_name,
        "w_primary": w_primary,
        "w_dinov2": w_dino,
        "w_clip_senet": w_clip_senet,
        "time_min": round(elapsed_min, 2),
        "returncode": result.returncode,
        "num_clusters": count_clusters(run_name),
        **metrics,
    }
    fusion_results.append(row)
    print(f"[{label}] returncode={result.returncode} time={elapsed_min:.1f} min metrics={metrics}")
    if result.returncode != 0:
        print("stdout tail:")
        print("\n".join(result.stdout.splitlines()[-40:]))
        print("stderr tail:")
        print("\n".join(result.stderr.splitlines()[-40:]))
        raise SystemExit(result.returncode)

results_path = Path("/kaggle/working/fusion_results.json")
payload = {
    "fusion_granularity": "tracklet",
    "baseline_note": "w_clip_senet=0.0 keeps the 10c v15 TransReID+DINOv2 score fusion control with w_dinov2=0.60.",
    "clip_senet_tracklet_embeddings": str(CLIP_TRACKLET_EMBEDDINGS_PATH),
    "results": fusion_results,
}
results_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("\nFusion sweep summary")
print(f"{'w_cs':>6} {'IDF1':>10} {'MOTA':>10} {'HOTA':>10} {'confl':>8} {'clusters':>9} {'w_dino':>8}")
for row in fusion_results:
    print(
        f"{row['w_clip_senet']:6.2f} "
        f"{str(row.get('mtmc_idf1')):>10} "
        f"{str(row.get('mota')):>10} "
        f"{str(row.get('hota')):>10} "
        f"{str(row.get('conflations')):>8} "
        f"{str(row.get('num_clusters')):>9} "
        f"{row['w_dinov2']:8.2f}"
    )

best = max(fusion_results, key=lambda row: row.get("mtmc_idf1") if row.get("mtmc_idf1") is not None else -1.0)
print(f"\nBest: w_cs={best['w_clip_senet']:.2f}, MTMC_IDF1={best.get('mtmc_idf1')}, saved to {results_path}")

## 7. Save Artifact Summary

In [ ]:
summary = {
    "run_name": RUN_NAME,
    "fusion_granularity": "tracklet",
    "inputs": {
        "transreid_primary": str(PRIMARY_EMBEDDINGS_PATH),
        "dinov2_tertiary": str(TERTIARY_DINOV2_PATH),
        "clip_senet_per_detection": str(CLIP_FEATURE_DIR),
        "clip_senet_tracklet": str(CLIP_TRACKLET_EMBEDDINGS_PATH),
    },
    "alignment_report": alignment_report,
    "weights": WEIGHT_SWEEP,
    "results_file": "/kaggle/working/fusion_results.json",
}
summary_path = Path("/kaggle/working/13d_summary.json")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))